# Culvert points to line segments workflow (optional step before hydro-conditioning)

## Input Datasets for this Notebook

Prior to running this notebook, put your culvert point shapefile (and the associated .dbf, .prj, .shx files) into the "Data" folder at the root project directory.

## 0. Import required modules

In [45]:
# System
import os
from pathlib import Path
import shutil
import sys
from netCDF4 import Dataset

# Ensure PWA directory is in the system path
prj_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if prj_path not in sys.path:
    sys.path.insert(0, prj_path)

# Analysis
import numpy as np
import pandas as pd
import geopandas as gpd
from sklearn.neighbors import NearestNeighbors
from shapely.geometry import LineString

# Custom Libraries
import src.py.phydap as phydap
from src.py.ui import *

# Visualization
import matplotlib.pyplot as plt

## 1. Organize data folders and files

### 1.1 Create subfolder containing culvert files

In [ ]:
# Set the name of the folder with your culvert files
CULVERT_FOLDER_NAME = usr_input().string("watershed name", default_value='temp_deadhorse')


# Path for this notebook
NOTEBOOK_PATH = Path.cwd()


# Path for PWA root
PWA_PATH = str(NOTEBOOK_PATH.parent.as_posix())


# Path for base raw data folder
BS_DATA_PATH = PWA_PATH + r"/Data/"


# Specify path for watershed folder
CULVERT_PATH = BS_DATA_PATH + CULVERT_FOLDER_NAME

### 1.2 Create subfolders in culvert subfolder ("Raw", "Interim", "Processed")

In [8]:
# Specify paths for depression depths subfolders
CUL_RAW_PATH = CULVERT_PATH + r"/Raw/"
CUL_INTERIM_PATH = CULVERT_PATH + r"/Interim/"
CUL_PROCESSED_PATH = CULVERT_PATH + r"/Processed/"


# Create watershed subfolders in specified paths
SUBFOLDERS_LIST = [CUL_RAW_PATH,
                   CUL_INTERIM_PATH,
                   CUL_PROCESSED_PATH]
for sub in SUBFOLDERS_LIST:
    os.makedirs(sub, exist_ok=True)

### 1.3 Move input data files to "Raw" subfolder

In [9]:
# Specify source and destination folders before moving files
src = Path(BS_DATA_PATH)
dst = Path(CUL_RAW_PATH)


# Move files in base data folder to watershed folder
for file in src.iterdir():
     if file.is_file():
        destination_file = dst / file.name
        if destination_file.exists():
            print(f"Skipped (already exists): {file.name}")
            continue
        shutil.move(str(file), destination_file)

## 2. Culvert points to lines workflow

### 2.1 Project the input file to metric CRS and inspect

In [ ]:
# Load the culverts shapefile
gdf = gpd.read_file(str(dst) + "/culverts.shp")


# Basic inspection
print(gdf.head())
print(gdf.columns)
print(gdf.crs)
print(len(gdf))

In [ ]:
# Project the shapefile to a metric CRS
gdf_proj = gdf.to_crs(epsg=32614) # This is UTM zone 14N (EPSG:32614), appropriate for Manitoba


# Confirm the crs projection
print("The projected CRS is:", 
      gdf_proj.crs)


# Inspect the geometry values
print("\nProjected geometry values:", 
      gdf_proj.geometry.head())


# Check two nearby points and their distance in the projected CRS to confirm the projection is working as expected
print("\nDistance between two nearby points in the projected CRS:", 
      gdf_proj.geometry.iloc[1].distance(gdf_proj.geometry.iloc[2]))


print("\nProceed to the next cell if the output makes sense.")

### 2.2 Find the nearest neighbors (within 30 metres)

In [ ]:
# Extract xy coordinates
coords = np.array(list(zip(gdf_proj.geometry.x, gdf_proj.geometry.y)))
print(coords)

In [ ]:
# Fit nearest neighbors model (2 neighbors: itself + nearest other)
nn = NearestNeighbors(n_neighbors=2, 
                      algorithm="kd_tree")
nn.fit(coords)
distances, indices = nn.kneighbors(coords)


# Print some results for the first index
print("For point with index zero,\nthe index for the point itself: ", indices[0][0], 
      "\nthe index for the nearest neighbor: ", indices[0][1],
      "\nand distance between them: ", distances[0][1])

In [ ]:
# Build a table of point pairs and their distances
pairs = []
for i in range(len(gdf_proj)):
    pt_id_left = i
    pt_id_right = indices[i][1]
    dist = distances[i][1]
    pairs.append((pt_id_left, pt_id_right, dist))


pairs_df = pd.DataFrame(pairs, columns=["pt_id_left", "pt_id_right", "dist"])


print(pairs_df.head())
print("The length of the pairs DataFrame is: ", len(pairs_df))
print("The min distance between points is: ", pairs_df["dist"].min())
print("The max distance between points is: ", pairs_df["dist"].max())
print("Proceed to the next cell if the outputs make sense. The maximum distance might be large if there are some isolated points, but most distances should be small if the culverts are generally close to each other.")

In [ ]:
# Only keep pairs that are within 30 metres of each other
threshold = 30 # metres
pairs_30m = pairs_df[pairs_df["dist"] <= threshold].copy()

print("The length of the pairs DataFrame (within 30m) is: ", len(pairs_30m))
print("The min distance between points (within 30m) is: ", pairs_30m["dist"].min())
print("The max distance between points (within 30m) is: ", pairs_30m["dist"].max())
print("The average distance between points (within 30m) is: ", pairs_30m["dist"].mean())
print(pairs_30m.head())

In [ ]:
# Here we remove duplicate pairs (aka 0-50 and 50-0 are the same pair)

# Create a unique key for each pair, with the smaller index first (e.g., 0, 50)
pairs_30m["pair_key"] = pairs_30m.apply(
    lambda row: tuple(sorted([row["pt_id_left"], row["pt_id_right"]])),
    axis=1
)


# Drop duplicate pairs based on the pair_key
pairs_unique = pairs_30m.drop_duplicates(subset="pair_key")


print("Proceed to the next cell if the outputs make sense. The length of the unique pairs DataFrame should be less than the length of the pairs_30m DataFrame (about half the length),\nand the pair keys should show unique combinations of point indices.\n")
print("The length of the unique pairs DataFrame is: ", len(pairs_unique))
print("Here is a sample of pair keys (the unique pairs of points that are within 30m of each other):")
pairs_unique.sample(5)

### 2.3 Build the line segments between each nearest-neighbor pair

In [ ]:
# Build the line segments between paired points
lines = []
for _, row in pairs_unique.iterrows():
    pt1 = gdf_proj.loc[row["pt_id_left"], "geometry"]
    pt2 = gdf_proj.loc[row["pt_id_right"], "geometry"]
    
    line = LineString([pt1, pt2])
    
    lines.append({
        "pt_id_left": row["pt_id_left"],
        "pt_id_right": row["pt_id_right"],
        "dist": row["dist"],
        "geometry": line
    })


lines_gdf = gpd.GeoDataFrame(lines, crs=gdf_proj.crs)
print("This cell will give us a GeoDataFrame where each row is one culvert pair and the geometry is a linestring connecting the two culverts.\nThe distance column shows how far apart the paired culverts are in metres (since we are in a projected CRS).\n")
print(lines_gdf.head())
print("\nThe length of the lines GeoDataFrame is: ", len(lines_gdf))

In [ ]:
# Write this into a shapefile so can be inspected closely on ArcGIS pro or some other software
lines_gdf.to_file(CUL_INTERIM_PATH + "culvert_pairs_lines.shp")

### 2.4 Join attributes from both points into the line segment

In [ ]:
# Here we join attributes from both points in each pair


# drop geometry so we don’t overwrite line geometry
attrs = gdf_proj.drop(columns="geometry")


# Left attributes
attrs_left = attrs.add_suffix("_left")


# Right attributes
attrs_right = attrs.add_suffix("_right")


# Join left point attributes
lines_gdf = lines_gdf.merge(attrs_left,
                            left_on="pt_id_left",
                            right_on="pt_id_left",
                            how="left"
                            )


# Join right point attributes
lines_gdf = lines_gdf.merge(attrs_right,
                            left_on="pt_id_right",
                            right_on="pt_id_right",
                            how="left"
                            )

print(lines_gdf.head())

### 2.5 Export the output lines shapefile

In [63]:
# Need to shorten column names before writing to shapefile, otherwise they will get truncated
rename_dict = {
    "pt_id_left": "pt_id_l",
    "pt_id_right": "pt_id_r",
    "diameter_m_left": "diam_l",
    "diameter_m_right": "diam_r",
    "length_m_left": "len_l",
    "length_m_right": "len_r",
    "easting_left": "east_l",
    "easting_right": "east_r",
    "northing_left": "north_l",
    "northing_right": "north_r",
    "elevation_left": "elev_l",
    "elevation_right": "elev_r"
}
lines_gdf_clean = lines_gdf.rename(columns=rename_dict)

# Only keep the relevant attribute columns for the final output
lines_gdf_clean = lines_gdf_clean[["pt_id_l", "pt_id_r", "dist", 
                                     "diam_l", "diam_r", "len_l", "len_r",
                                     "east_l", "north_l", "elev_l",
                                     "east_r", "north_r", "elev_r",
                                     "geometry"]]

In [ ]:
# Some extra checks before writing the final output
print("Here, make sure that the ids and the geometry isn't missing (i.e., there are zero NaN values):")
print(lines_gdf_clean.isna().sum())

# Confirm no duplicates once again (should be zero)
print(lines_gdf_clean[["pt_id_l", "pt_id_r"]].duplicated().sum())
print(lines_gdf_clean[["pt_id_l", "pt_id_r"]].duplicated(keep=False).sum())


# Confirm valid geometries (everything should be True)
print(lines_gdf_clean.geometry.is_valid.value_counts())

In [64]:
# Write the final lines GeoDataFrame with attributes to a shapefile so can be inspected closely on ArcGIS pro or some other software (and also fed into hydroconditioning script!)
lines_gdf_clean.to_file(CUL_PROCESSED_PATH + "culvert_pairs.shp")